# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset—specifically, mass spectrometry analysis protein data—using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema and loaded directly using its URL.

**Citation**: Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print a summary
print(f"Dataset loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
from pprint import pprint

# List all record sets and their fields using @id
print("All record sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print("- Record set name:", getattr(rs, 'name', None))
    print("  @id:", rs.id)
    print("  Description:", getattr(rs, 'description', None))
    print("  Fields and columns:")
    for field in rs.fields:
        print(f"    - Field name: {getattr(field, 'name', None)} | @id: {field.id}")
        if hasattr(field, 'column') and field.column:
            print(f"      • column @id: {field.column.id}")
    print("  ---")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Each entity (`record set`, `field`, `column`) is referenced by its `@id`.

In [ ]:
# Prepare to collect dataframes for all record sets
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

print(f"Extracting data for record sets: {record_set_ids}")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    try:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nLoaded DataFrame for {record_set_id} with shape {dataframes[record_set_id].shape}")
        print(dataframes[record_set_id].head(2))
    except Exception as exc:
        print(f"\nNo tabular data for {record_set_id}: {exc}")

## 4. Exploratory Data Analysis (EDA)
We perform example filtering, normalization, and grouping. First, choose the main protein table by its `@id`, pick a numeric field `@id`, and a groupable/categorical field.

In [ ]:
# Set your primary record set and field IDs by inspecting cell output above
# You may need to adjust these if the schema changes in future dataset versions

# Example assignments for this dataset (change as needed based on printed overview above):
# Suppose main record set is 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/protein_recordset',
# and has a numeric field 'coverage_percentage' and groupable field 'sample_type'.

# Let's infer the likely IDs if present; otherwise, please use notebook outputs from previous cell
main_record_set_id = record_set_ids[0]  # Use the first record set present
df = dataframes[main_record_set_id]

print(f"Columns in DataFrame ({main_record_set_id}):\n{df.columns.tolist()}")

# Choose a numeric field and group field by inspecting columns
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'coverage' in col.lower() or 'abund' in col.lower():
        numeric_field_id = col
    if 'sample' in col.lower() or 'type' in col.lower() or 'group' in col.lower():
        group_field_id = col
if numeric_field_id is None:
    print("No obvious numeric field found. Using the first column.")
    numeric_field_id = df.columns[0]
if group_field_id is None:
    print("No obvious group field found. Using the second column if present.")
    group_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
print(f"\nSelected numeric field: {numeric_field_id} (threshold: {threshold})")
print(f"Selected group field: {group_field_id}\n")

# Filter by numeric threshold (e.g., above mean)
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
else:
    # try coercing to numeric
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()

print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize the field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
else:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Groupby if field present and not too unique
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field_id} (showing first 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Plot distributions and relationships of the main numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by group field
if group_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated step-by-step how to use the FAIR² Croissant metadata and the `mlcroissant` library to:
- Discover record sets and fields (`@id` usage throughout ensures robust referencing)
- Load mass spectrometry protein/EV tabular data from Croissant
- Filter, normalize, and group numeric results
- Visualize distributions and group differences

**Next steps**: You can extend this workflow by sampling additional fields, cleaning further, or exporting for ML analysis. Always check and cite the dataset as described in cell 1.